# Lab 3 — Notebook 02: Learning with Gradient Descent

**Course:** Artificial Intelligence with Python  
**Estimated duration:** 60 minutes  
**Prerequisites:** Lab 3 Notebook 01 (models, predictions, MAE and MSE)  
**Libraries:** NumPy, Pandas, Matplotlib — *no scikit-learn training methods*

## Learning objectives

By the end of this notebook, you should be able to:

- Explain why training means **adjusting model parameters to reduce loss**.
- Describe the roles of the **gradient**, **learning rate**, and **iterations (epochs)** (— full passes through the training data, also called iterations in this notebook).
- Implement a simple gradient-descent training loop using NumPy.
- Visualize changes in model predictions and training loss.
- Identify learning rates that are too small, suitable, or unstable.

**How to work:** Run examples in order; write your own solutions in the `TODO` cells and short reflections in the Markdown prompts. The notebook is designed to run from top to bottom even before you complete the exercises.

> **Important:** This lab uses the same small synthetic apartment dataset as Notebook 01. We are exploring the *learning mechanism* here, not evaluating generalization. Train/validation/test behavior is the topic of Notebook 03.

## 1. Reload the dataset and prepare a numerical model (10 min)

We again predict apartment rent from floor area with a linear model:

$\hat{y} = wx+b$.

A large input (e.g. 75 m²) and output (e.g. €1,300) make learning-rate experiments hard to interpret. To make the gradient-descent updates easier to follow, we will use:

- `z`: floor area measured in **standard deviations from the mean**;
- `t`: monthly rent measured in **thousands of euros**.

The model trained in this notebook is therefore \(\hat{t}=wz+b\). This changes the numerical scale, **not** the prediction task. When reporting rent, multiply predictions by 1,000.

In this notebook, we use all 72 observations to demonstrate optimization. When evaluating a real model, scaling parameters must be fitted on **training data only**, as practised in Lab 2.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

csv_path = Path("data/linear_rent.csv")
if not csv_path.exists():
    raise FileNotFoundError(
        "Cannot find data/linear_rent.csv. "
        "Open JupyterLab in the lab03 folder and use the included CSV."
    )

df = pd.read_csv(csv_path)
print("Rows:", len(df))
display(df.head())

area = df["area_m2"].to_numpy(dtype=float)
rent_eur = df["monthly_rent_eur"].to_numpy(dtype=float)

area_mean = area.mean()
area_std = area.std()
z = (area - area_mean) / area_std
t = rent_eur / 1000.0

print(f"Area mean: {area_mean:.2f} m²; standard deviation: {area_std:.2f} m²")
print(f"Standardized areas: mean {z.mean():.4f}, std {z.std():.4f}")
print(f"Rent is now measured in thousands of euros: mean {t.mean():.3f}")

In [ ]:
def predict(z_values, weight, bias):
    """Predict rent in thousands of euros."""
    return weight * z_values + bias


def mse(actual, predicted):
    """Return mean squared error in (thousands of euros) squared."""
    return float(np.mean((actual - predicted) ** 2))

initial_weight = 0.0
initial_bias = 0.0
initial_predictions = predict(z, initial_weight, initial_bias)
print(f"Initial MSE: {mse(t, initial_predictions):.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(area, rent_eur, alpha=0.7, label="Observed apartment rents")
ax.axhline(0, linestyle="--", label="Initial model prediction")
ax.set(xlabel="Area (m²)", ylabel="Monthly rent (€)",
       title="Before training: w = 0, b = 0")
ax.legend()
plt.show()

### Exercise 1 — Check the starting point (3 min)

1. Calculate the model's initial predictions when `weight = 0` and `bias = 1.0`. What rent in euros does that represent?
2. Compute its MSE. Is it lower than the original `weight = 0, bias = 0` model?
3. Why is a constant prediction unable to capture the relationship between floor area and rent?

In [ ]:
# TODO: predict using weight = 0 and bias = 1.0
# TODO: compute the MSE and compare with the original model

**Your observation:** TODO — Explain what changing the bias does when the weight is zero.

## 2. Understand the gradient and one learning step (15 min)

The **gradient** tells us how loss changes when we change the model parameters. For a linear model with mean squared error

\begin{equation*}
L(w,b)=\frac{1}{n}\sum_{i=1}^n(wz_i+b-t_i)^2,
\end{equation*}

the gradients are:

\begin{equation*}
\frac{\partial L}{\partial w}=\frac{2}{n}\sum_{i=1}^n (\hat{t}_i-t_i)z_i,
\quad
\frac{\partial L}{\partial b}=\frac{2}{n}\sum_{i=1}^n (\hat{t}_i-t_i).
\end{equation*}

You do **not** need to derive these formulas in this lab. Focus on what they do: they indicate a direction in which loss increases. Gradient descent moves in the **opposite** direction.

\begin{equation*}
w_{\mathrm{new}}=w-\alpha\frac{\partial L}{\partial w}, \qquad
b_{\mathrm{new}}=b-\alpha\frac{\partial L}{\partial b}.
\end{equation*}

Here \(\alpha\) (alpha) is the **learning rate**: it controls the size of each update.

In [ ]:
def calculate_gradients(z_values, actual, weight, bias):
    """Return gradients of mean squared error with respect to w and b."""
    estimated = predict(z_values, weight, bias)
    error = estimated - actual
    gradient_w = 2 * np.mean(error * z_values)
    gradient_b = 2 * np.mean(error)
    return float(gradient_w), float(gradient_b)

weight, bias = 0.0, 0.0
learning_rate = 0.10
loss_before = mse(t, predict(z, weight, bias))
grad_w, grad_b = calculate_gradients(z, t, weight, bias)

new_weight = weight - learning_rate * grad_w
new_bias = bias - learning_rate * grad_b
loss_after = mse(t, predict(z, new_weight, new_bias))

print(f"Gradient: dw={grad_w:.4f}, db={grad_b:.4f}")
print(f"Parameters before: w={weight:.4f}, b={bias:.4f}")
print(f"Parameters after:  w={new_weight:.4f}, b={new_bias:.4f}")
print(f"MSE: {loss_before:.4f} -> {loss_after:.4f}")

### Exercise 2 — Perform an update manually (5 min)

1. Start with `weight = 0`, `bias = 0` and `learning_rate = 0.05`.
2. Use `calculate_gradients()` to compute the gradients.
3. Perform **one** parameter update using NumPy/Python code, and compare the MSE before and after.
4. Does each parameter necessarily increase? Explain why the **sign** of a gradient matters.

*Hint:* Use `weight - learning_rate * grad_w` and the corresponding expression for the bias.

In [ ]:
# TODO: start from weight = 0.0 and bias = 0.0
# TODO: calculate the gradients
# TODO: update both parameters with learning_rate = 0.05
# TODO: compare MSE before and after the update

**Interpretation:** TODO — What does the gradient tell you about which way to move the parameters?

## 3. Implement iterative learning (15 min)

One update is not enough. During training we repeat the same steps:

1. Predict using the current parameters.
2. Calculate prediction errors and gradients.
3. Update the parameters.
4. Record the loss.
5. Repeat for the requested number of **iterations**.

Because we use the **entire dataset for each update**, this is *batch gradient descent*. In many deep-learning frameworks, training also uses mini-batches; we will encounter that later.

The history contains the loss **after** each update and also the initial loss (iteration 0).

In [ ]:
def train_gradient_descent(z_values, actual, learning_rate=0.1,
                           iterations=40, initial_weight=0.0, initial_bias=0.0):
    """Train a linear model by batch gradient descent using NumPy."""
    weight = float(initial_weight)
    bias = float(initial_bias)
    history = [mse(actual, predict(z_values, weight, bias))]

    for _ in range(iterations):
        grad_w, grad_b = calculate_gradients(z_values, actual, weight, bias)
        weight -= learning_rate * grad_w
        bias -= learning_rate * grad_b
        history.append(mse(actual, predict(z_values, weight, bias)))

    return weight, bias, np.array(history)

trained_w, trained_b, loss_history = train_gradient_descent(
    z, t, learning_rate=0.1, iterations=40
)
print(f"Learned parameters: w = {trained_w:.4f}, b = {trained_b:.4f}")
print(f"MSE: {loss_history[0]:.5f} -> {loss_history[-1]:.5f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(len(loss_history)), loss_history, marker="o", markersize=3)
ax.set(xlabel="Iteration", ylabel="MSE (thousands of €)²",
       title="Training loss decreases during gradient descent")
ax.grid(alpha=0.25)
plt.show()

In [ ]:
# Compare model predictions before and after learning.
area_order = np.argsort(area)
final_rent_eur = predict(z, trained_w, trained_b) * 1000

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(area, rent_eur, alpha=0.65, label="Observed rents")
ax.plot(area[area_order], final_rent_eur[area_order],
        linewidth=2, label="Model after training")
ax.set(xlabel="Apartment area (m²)", ylabel="Monthly rent (€)",
       title="A learned straight line")
ax.legend()
plt.show()

# Translate our standardized model back to familiar units:
# rent (€) = euro_per_m2 * area_m2 + intercept_eur
euro_per_m2 = 1000 * trained_w / area_std
intercept_eur = 1000 * (trained_b - trained_w * area_mean / area_std)
print(f"Equivalent model: rent ≈ {euro_per_m2:.2f} × area_m2 + {intercept_eur:.2f} €")

### Exercise 3 — Observe convergence (5 min)

1. Train the model for **5**, **15**, and **40** iterations, all with a learning rate of `0.1`.
2. Print the final MSE for each run.
3. Why does the model benefit from more updates initially, but improve less and less later?

*Hint:* Call `train_gradient_descent()` with a different `iterations` value each time.

In [ ]:
# TODO: train for 5, 15, and 40 iterations
# TODO: print the final MSE from each returned loss history

**Your observation:** TODO — Describe what the loss curve says about learning progress.

## 4. Experiment with learning rates (15 min)

The learning rate affects how quickly—or whether—the model converges:

- **Too small:** learning proceeds slowly.
- **Suitable:** loss falls quickly and stabilizes.
- **Too large:** updates can overshoot the minimum and the loss can oscillate or increase.

The next plot compares three rates using the **same initial parameters** and 30 iterations. A logarithmic y-axis makes both small and very large losses visible. These numerical examples are specific to our standardized input and scaled target.

In [ ]:
rates = [0.005, 0.1, 1.1]
fig, ax = plt.subplots(figsize=(7, 4))

for rate in rates:
    w_rate, b_rate, history = train_gradient_descent(
        z, t, learning_rate=rate, iterations=30
    )
    print(f"Rate {rate:>5}: start MSE={history[0]:.4f}, "
          f"final MSE={history[-1]:.4f}")
    ax.plot(range(len(history)), history, label=f"learning rate = {rate}")

ax.set(xlabel="Iteration", ylabel="MSE (log scale)",
       title="Effect of learning rate on training")
ax.set_yscale("log")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

### Exercise 4 — Find an effective learning rate (5 min)

1. Test at least **three more** learning rates in the range `0.01` to `0.8`.
2. For each, train for **20 iterations** and print the final MSE.
3. Which tested rate reaches the lowest loss in this limited time?
4. Is your answer guaranteed to remain optimal for another dataset or a different feature scale? Why or why not?

In [ ]:
# TODO: choose at least three learning rates
# TODO: train each model for 20 iterations
# TODO: compare final losses and justify your selection

**Your observation:** TODO — Explain how the learning rate changes the training behavior.

## 5. Mini-challenge and reflection (5 min)

**Mini-challenge: Train a model and predict an unseen apartment.**

Choose a learning rate and iteration count that produce a stable, low-loss model. Then:

1. Train the model from `weight = 0` and `bias = 0`.
2. Predict the monthly rent of a **75 m² apartment**. Remember to apply the same area standardization and convert from thousands of euros back to euros.
3. Report your hyperparameters and final training MSE.
4. Explain why a small *training* MSE is **not** proof that the model will predict unseen apartments accurately.

**Stretch task (optional):** Modify `train_gradient_descent()` to store a weight history and draw how `w` changes as training proceeds.

In [ ]:
# TODO: choose a learning rate and number of iterations
# TODO: train_gradient_descent(...) with your settings
# TODO: standardize a 75 m² apartment using area_mean and area_std
# TODO: predict its rent in euros
# TODO: print your final training loss

**Your reflection:**

- **Chosen learning rate:** TODO
- **Chosen iterations:** TODO
- **Final training MSE:** TODO
- **Predicted rent for 75 m²:** TODO
- **Why low training loss is insufficient:** TODO

### Key takeaways

- Training changes the model's parameters to reduce the loss.
- The gradient indicates how to adjust each parameter; the learning rate controls update size.
- A learning curve helps diagnose slow learning and unstable updates.
- Numerical scaling changes which learning rates work well.
- In **Notebook 03 — Generalization**, we will learn why training loss and performance on new data can differ.